# Libraries

In [1]:
import sys
import os
from sklearn.metrics import precision_score, recall_score, f1_score
import json, re
import numpy as np
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import precision_score, recall_score, f1_score

In [2]:
this_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(this_dir, os.pardir))
if project_root not in sys.path:
    sys.path.insert(0, project_root)
os.chdir(project_root)

In [3]:
from kg_rag.util import *

In [4]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

In [5]:

print(openai.__version__)

1.61.0


# GeneTuring Benchmarks

In [6]:
geneturing_data = pd.read_csv("data/geneTuring/Q&A_dataset.csv")
geneturing_data.columns

Index(['Model', 'Module', 'Question', 'Goldstandard', 'Unnamed: 4',
       'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7'],
      dtype='object')

In [7]:
geneturing_data["Module"].unique()

array(['Amino acid translation', 'DNA sequence extraction', 'Gene alias',
       'Gene disease association', 'Gene location', 'Gene ontology',
       'Human genome DNA aligment',
       'Human genome DNA aligment programming',
       'Multi-species DNA aligment',
       'Multi-species DNA aligment programming', 'Gene name conversion',
       'Gene name extraction', 'Protein-coding genes',
       'Gene SNP association', 'SNP location', 'TF regulation'],
      dtype=object)

In [8]:
geneturing_data.shape

(1600, 8)

In [9]:
def extract_disease_from_question(question: str) -> str | None:
    """
    Extracts the disease name from a question of the form:
      'The name of the gene related to <disease> is'
    
    Works case-insensitively and trims whitespace/punctuation.
    Returns the extracted disease string, or None if no match found.
    """
    if not isinstance(question, str):
        return None

    # regex: capture text between "related to" and "is"
    m = re.search(r'related to (.+?) is\b', question, flags=re.IGNORECASE)
    if m:
        disease = m.group(1).strip(" .,:;!?")
        return disease
    return None

In [10]:
gene_dis_data = geneturing_data[geneturing_data["Module"].str.strip() == "Gene disease association"]
gene_dis_data = gene_dis_data.iloc[:, :-4]
gene_dis_data.head(1)

,Model,Module,Question,Goldstandard
300,GeneGPT,Gene disease association,The name of the gene related to Hemolytic anemia due to phosphofructokinase deficiency is,PFKL


In [11]:
gene_dis_data["extracted_disease"] = gene_dis_data["Question"].apply(extract_disease_from_question)


In [12]:
gene_dis_data.tail()

,Model,Module,Question,Goldstandard,extracted_disease
395,GeneGPT,Gene disease association,The name of the gene related to Tn polyagglutination syndrome is,C1GALT1C1,Tn polyagglutination syndrome
396,GeneGPT,Gene disease association,The name of the gene related to Congenital contractures of the limbs and face is,NALCN,Congenital contractures of the limbs and face
397,GeneGPT,Gene disease association,The name of the gene related to Hartsfield syndrome is,FGFR1,Hartsfield syndrome
398,GeneGPT,Gene disease association,The name of the gene related to Hodgkin lymphoma is,KLHDC8B,Hodgkin lymphoma
399,GeneGPT,Gene disease association,The name of the gene related to Hemorrhagic diathesis due to antithrombin Pittsburgh is,SERPINA1,Hemorrhagic diathesis due to antithrombin Pittsburgh


In [13]:
gene_dis_data["extracted_in_question"] = gene_dis_data.apply(
    lambda row: (
        isinstance(row["extracted_disease"], str)
        and row["extracted_disease"].strip().lower()
        in row["Question"].lower()
    ),
    axis=1
)

In [14]:
n_total = len(gene_dis_data)
n_found = gene_dis_data["extracted_in_question"].sum()
n_not_found = n_total - n_found

print("==== Extracted Disease in Question Check ====")
print(f"Total rows:                 {n_total}")
print(f"Extracted disease in question: {n_found}")
print(f"Not found in question:      {n_not_found}")
print(f"Proportion found:           {n_found / n_total:.3f}")

==== Extracted Disease in Question Check ====
Total rows:                 100
Extracted disease in question: 100
Not found in question:      0
Proportion found:           1.000


In [16]:
# gene_dis_data[gene_dis_data["extracted_in_question"]==False]

In [18]:
gene_dis_data.head(2)

,Model,Module,Question,Goldstandard,extracted_disease,extracted_in_question
300,GeneGPT,Gene disease association,The name of the gene related to Hemolytic anemia due to phosphofructokinase deficiency is,PFKL,Hemolytic anemia due to phosphofructokinase deficiency,True
301,GeneGPT,Gene disease association,The name of the gene related to Distal renal tubular acidosis is,"SLC4A1, ATP6V0A4",Distal renal tubular acidosis,True


In [33]:
# BTE curies

In [58]:
def get_curie_id(name: str, retries: int = 3, sleep: float = 2.0) -> str | None:
    """
    Query the Translator Name Resolver API for a CURIE ID, with retry and timeout handling.
    """
    url = "https://name-lookup.ci.transltr.io/lookup"
    params = {"string": name, "offset": 0, "limit": 1}

    for attempt in range(retries):
        try:
            r = requests.get(url, params=params, timeout=20)
            r.raise_for_status()
            data = r.json()

            if data and isinstance(data, list) and "curie" in data[0]:
                return data[0]["curie"]

        except (requests.exceptions.Timeout, requests.exceptions.ConnectionError) as e:
            print(f"Timeout for '{name}', attempt {attempt+1}/{retries}. Retrying...")
            time.sleep(sleep)

        except Exception as e:
            print(f"Error for {name}: {e}")
            return None

    return None 

In [59]:
unique_diseases = gene_dis_data["extracted_disease"].dropna().unique()

# Query for eaach id
curie_mapping = []
for name in tqdm(unique_diseases, desc="Fetching CURIEs"):
    curie = get_curie_id(name)
    curie_mapping.append({"extracted_disease": name, "curie_id": curie})

# Create DataFrame
curie_df = pd.DataFrame(curie_mapping)


Fetching CURIEs: 100%|██████████████████████████████| 100/100 [00:58<00:00,  1.71it/s]


In [60]:
merged_df = gene_dis_data.merge(curie_df, on="extracted_disease", how="left")

In [61]:
merged_df.head()

,Model,Module,Question,Goldstandard,extracted_disease,curie_id
0,GeneGPT,Gene disease association,The name of the gene related to Hemolytic anemia due to phosphofructokinase deficiency is,PFKL,Hemolytic anemia due to phosphofructokinase deficiency,MONDO:0009113
1,GeneGPT,Gene disease association,The name of the gene related to Distal renal tubular acidosis is,"SLC4A1, ATP6V0A4",Distal renal tubular acidosis,MONDO:0015827
2,GeneGPT,Gene disease association,The name of the gene related to Pseudohypoparathyroidism Ic is,GNAS,Pseudohypoparathyroidism Ic,MONDO:0019992
3,GeneGPT,Gene disease association,The name of the gene related to Glycine N-methyltransferase deficiency is,GNMT,Glycine N-methyltransferase deficiency,MONDO:0011698
4,GeneGPT,Gene disease association,The name of the gene related to Meesmann corneal dystrophy is,"KRT12, KRT3",Meesmann corneal dystrophy,MONDO:0007379


In [62]:
merged_df["curie_id"].isna().value_counts()


curie_id
False    100
Name: count, dtype: int64

In [64]:
entity_df = merged_df[["curie_id", "extracted_disease"]].dropna().copy()
entity_df.columns = ["entity_id", "name"]
entity_df["type"] = "Disease"


entity_df = entity_df[["type", "entity_id", "name"]]
entity_df.head(2)

,type,entity_id,name
0,Disease,MONDO:0009113,Hemolytic anemia due to phosphofructokinase deficiency
1,Disease,MONDO:0015827,Distal renal tubular acidosis


In [ ]:
# get bte api response

In [65]:
def get_bte(entity_id):
    try:
        base_uri = "https://bte.transltr.io/v1/query"
        
        query = {
            "message": {
                "query_graph": {
                    "nodes": {
                        "n0": {
                            "categories": ["biolink:Disease"],
                            "ids": [entity_id]
                        },
                        "n1": {
                            "categories": ["biolink:Gene"]
                        }
                    },
                    "edges": {
                        "e01": {
                            "subject": "n0",
                            "object": "n1"
                        }
                    }
                }
            }
        }

        print(f"Querying bte context for {entity_id}...")
        result = get_bte_api_resp(base_uri, query)
        return result

    except Exception as e:
        print(f"Error querying {entity_id}: {e}")
        return None

In [66]:
def serialize_result(result):
    """Compress and serialize a JSON-compatible object."""
    return zlib.compress(json.dumps(result).encode())

def deserialize_result(serialized_result):
    """Decompress and deserialize the stored context."""
    return json.loads(zlib.decompress(serialized_result).decode())

In [67]:
def process_entities(df, results_filepath="data/geneTuring/gene_disease_assoc_bte_results.parquet"):
    """
    Process gene entities to retrieve homolog data via BTE and store results.
    
    Parameters:
      - df: DataFrame with columns ['type', 'entity_id', 'name']
      - results_filepath: where to read/write the results (Parquet file)
      
    Returns:
      - Updated results DataFrame
    """
    try:
        results_df = pd.read_parquet(results_filepath)
    except FileNotFoundError:
        os.makedirs(os.path.dirname(results_filepath), exist_ok=True)
        results_df = pd.DataFrame(columns=["entity_id", "name", "type", "context"])

    processed_entity_ids = set(results_df["entity_id"])

    for _, row in df.iterrows():
        entity_id = row["entity_id"]
        entity_type = row["type"]
        name = row["name"]

        if pd.isna(entity_id) or entity_id in processed_entity_ids:
            continue

        context = get_bte(entity_id)  
        serialized_context = serialize_result(context) if context else None

        new_row = pd.DataFrame({
            "entity_id": [entity_id],
            "name": [name],
            "type": [entity_type],
            "context": [serialized_context]
        })

        results_df = pd.concat([results_df, new_row], ignore_index=True)
        processed_entity_ids.add(entity_id)

        
        results_df.to_parquet(results_filepath, compression="snappy")

    return results_df

In [68]:
entity_df.shape

(100, 3)

In [69]:
entity_df.head(1)

,type,entity_id,name
0,Disease,MONDO:0009113,Hemolytic anemia due to phosphofructokinase deficiency


In [102]:
result_geneTuring_bte_context = process_entities(entity_df)

Querying bte context for MONDO:0019992...
Querying bte context for MONDO:0011698...
Querying bte context for MONDO:0007379...
Querying bte context for MONDO:0014528...
Querying bte context for MONDO:0011242...
Querying bte context for UMLS:C2747802...
Querying bte context for MONDO:0012559...
Querying bte context for MONDO:0008305...
Querying bte context for UMLS:C0393414...
Querying bte context for MONDO:0008163...
Querying bte context for MONDO:0010977...
Querying bte context for MONDO:0859085...
Querying bte context for MONDO:0008758...
Querying bte context for MONDO:0013892...
Querying bte context for MONDO:0018852...
Querying bte context for MONDO:0016543...
Querying bte context for MONDO:0013678...
Querying bte context for MONDO:0800030...
Querying bte context for UMLS:C1844831...
Querying bte context for MONDO:0032877...
Querying bte context for MONDO:0010094...
Querying bte context for MONDO:0009491...
Querying bte context for MONDO:0017734...
Querying bte context for MONDO:003

In [19]:
geneTuring_bte_context_parquet = pd.read_parquet("data/geneTuring/gene_disease_assoc_bte_results.parquet")
geneTuring_bte_context_parquet.shape

(100, 4)

In [20]:
geneTuring_bte_context_parquet.columns

Index(['entity_id', 'name', 'type', 'context'], dtype='object')

In [22]:
# context = deserialize_result(geneTuring_bte_context_parquet.loc[1, "context"])
# context

In [105]:
def get_bte_context(id):

    path = "data/geneTuring/gene_disease_assoc_bte_results.parquet"
    bte_df = pd.read_parquet(path)

    if id not in bte_df["entity_id"].values:
        print(f"ID {id} not found.")
        return []

    try:
        serialized = bte_df.loc[bte_df["entity_id"] == id, "context"].iloc[0]
        node_context = deserialize_result(serialized)
    except Exception as e:
        print(f"Deserialization failed for {id}: {e}")
        return []

    # Nodes and edges
    nodes = node_context.get("message", {}).get("knowledge_graph", {}).get("nodes", {})
    edges = node_context.get("message", {}).get("knowledge_graph", {}).get("edges", {})

    nbr_nodes = [
        (nid, ", ".join(info.get("categories", [])), info.get("name", ""))
        for nid, info in nodes.items()
    ]
    nbr_edges = []
    for eid, einfo in edges.items():
        predicate = einfo.get("predicate", "")
        subj = einfo.get("subject", "")
        obj = einfo.get("object", "")
        sources = next(
            (s.get("resource_id") for s in einfo.get("sources", []) if s.get("resource_role") == "primary_knowledge_source"),
            None
        )
        if not sources:
            sources = ", ".join(s.get("resource_id", "") for s in einfo.get("sources", []))
        nbr_edges.append((subj, predicate, obj, sources))

    # DataFrames
    df_nodes = pd.DataFrame(nbr_nodes, columns=["node_id", "categories", "node_name"])
    df_edges = pd.DataFrame(nbr_edges, columns=["subject", "predicate", "object", "sources"])
    if not df_edges.empty:
        df_edges["sources"] = df_edges["sources"].str.upper()

    # Merge and describe
    m1 = pd.merge(df_edges, df_nodes, left_on="subject", right_on="node_id").drop("node_id", axis=1)
    m1["subject_description"] = m1["categories"] + " " + m1["node_name"]
    m1.drop(["categories", "node_name"], axis=1, inplace=True)

    m2 = pd.merge(m1, df_nodes, left_on="object", right_on="node_id", how="left").drop("node_id", axis=1)
    m2["object_description"] = m2["categories"] + " " + m2["node_name"]
    m2.drop(["categories", "node_name"], axis=1, inplace=True)

    # Clean and readable
    m2 = m2[m2["object_description"] != ""]
    m2 = m2.map(make_readable)
    m2["subject_clean"] = m2["subject_description"].apply(clean_description)
    m2["object_clean"] = m2["object_description"].apply(clean_description)
    m2["context"] = m2.apply(
        lambda row: f"{row['subject_clean']} is {row['predicate']} {row['object_clean']}.",
        axis=1
    )

    # Group similar relations
    grouped = (
        m2.groupby(["subject_clean", "predicate"])["object_clean"]
        .apply(list)
        .reset_index()
    )
    grouped["object_list_str"] = grouped["object_clean"].apply(objects_to_text)
    grouped["context_grouped"] = grouped.apply(
        lambda row: f"{row['subject_clean']} is {row['predicate']} {row['object_list_str']}.",
        axis=1
    )

    final_df = grouped[["context_grouped"]].drop_duplicates()
    node_context_final = final_df["context_grouped"].tolist()

    return node_context_final


In [106]:
def retrieve_bte_context(
    question,
    id,
    name,
    embedding_function,
    context_sim_threshold=0,
    context_sim_min_threshold=0.5,
    context_volume=None
):
    print("Question:", question)
    question_embedding = embedding_function.embed_query(question)

    print("Processing ID:", id)
    context = get_bte_context(id)

    if context:
        embeddings = embedding_function.embed_documents(context)
        similarities = []

        for context_embedding in embeddings:
            score = cosine_similarity(
                np.array(question_embedding).reshape(1, -1),
                np.array(context_embedding).reshape(1, -1)
            )[0][0]
            similarities.append(score)

        # print("Similarity scores:", similarities)

        similarity_indices = sorted(
            [(sim, idx) for idx, sim in enumerate(similarities)],
            key=lambda pair: pair[0],
            reverse=True
        )

        percentile_threshold = np.percentile(
            [pair[0] for pair in similarity_indices],
            context_sim_threshold
        )

        high_similarity_indices = [
            idx for sim, idx in similarity_indices
            if sim >= percentile_threshold and sim >= context_sim_min_threshold
        ]

        if context_volume is not None:
            high_similarity_indices = high_similarity_indices[:context_volume]

        high_similarity_context = [context[idx] for idx in high_similarity_indices]

    else:
        print("No context found for id; skipping similarity.")
        high_similarity_context = []

    
    # Top contexts (no reformatting)
    node_context_extracted = "Context:\n" + "\n".join(high_similarity_context) + "\n"

    # Full context (no reformatting)
    combined_context_str = "Context:\n" + "\n".join(context) + "\n" if context else "Context:\n\n"

    return node_context_extracted, combined_context_str


In [107]:
sentence_embedding_model='pritamdeka/S-PubMedBert-MS-MARCO'
embedding_function = load_sentence_transformer(sentence_embedding_model)

In [108]:
def make_readable(text):
    # Use regular expressions for case-insensitive replacements
    text = re.sub(r"\bbiolink:gene\b", "", text, flags=re.IGNORECASE)  # precise removal
    text = re.sub(r"\bbiolink:disease\b", "", text, flags=re.IGNORECASE)
    text = re.sub(r"biolink:", "", text, flags=re.IGNORECASE)  # remove other biolink: prefixes
    text = re.sub(r"_", " ", text)
    text = re.sub(r":", "", text)
    text = re.sub(r"INFORES", "", text, flags=re.IGNORECASE)
    text = ' '.join(text.split())  # collapse whitespace
    return text

In [81]:
merged_df.head(1)

,Model,Module,Question,Goldstandard,extracted_disease,curie_id
0,GeneGPT,Gene disease association,The name of the gene related to Hemolytic anemia due to phosphofructokinase deficiency is,PFKL,Hemolytic anemia due to phosphofructokinase deficiency,MONDO:0009113


In [86]:
question = merged_df.loc[0, "Question"]
id = merged_df.loc[0, "curie_id"]
name = merged_df.loc[0, "extracted_disease"]

ctx_top, ctx_full = retrieve_bte_context(
    question=question,
    id=id,
    name=name,
    embedding_function=embedding_function  
)
# print(ctx_top)

Question: The name of the gene related to Hemolytic anemia due to phosphofructokinase deficiency is
Processing ID: MONDO:0009113
Similarity scores: [0.9358575221510892, 0.9439686337414468, 0.9485201774564322, 0.937604798736055]


In [87]:
ctx_top

'Context:\nhemolytic anemia due to diphosphoglycerate mutase deficiency is genetically associated with BPGM, BPGM, BPGM, BPGM.\nhemolytic anemia due to diphosphoglycerate mutase deficiency is condition associated with gene BPGM, GPM1, GPM2, GPM3, CG7059, Pglym87, Pglym78, Bpgm, Bpgm, bpgm, BPGM, BPGM.\nhemolytic anemia due to diphosphoglycerate mutase deficiency is related to BPGM.\nhemolytic anemia due to diphosphoglycerate mutase deficiency is caused by BPGM.\n'

In [88]:
ctx_full

'Context:\nhemolytic anemia due to diphosphoglycerate mutase deficiency is caused by BPGM.\nhemolytic anemia due to diphosphoglycerate mutase deficiency is condition associated with gene BPGM, GPM1, GPM2, GPM3, CG7059, Pglym87, Pglym78, Bpgm, Bpgm, bpgm, BPGM, BPGM.\nhemolytic anemia due to diphosphoglycerate mutase deficiency is genetically associated with BPGM, BPGM, BPGM, BPGM.\nhemolytic anemia due to diphosphoglycerate mutase deficiency is related to BPGM.\n'

In [110]:


def build_bte_context_df(
    df,
    embedding_function,
    save_path
):
    if os.path.exists(save_path):
        out_df = pd.read_csv(save_path)
        processed = set(out_df["Question"])
    else:
        out_df = pd.DataFrame(columns=[
            "Question", "extracted_disease", "curie_id", "Goldstandard",
            "node_context_extracted", "combined_context_str",
            "node_context_extracted_compressed", "combined_context_str_compressed"
        ])
        processed = set()

    for _, row in tqdm(df.iterrows(), total=len(df), desc="Processing id's"):
        question = row["Question_new"]
        name = row["extracted_disease"]
        id = row["curie_id"]
        gold = row.get("Goldstandard", None)

        if pd.isna(id) or question in processed:
            continue

        try:
            # Use similarity-filtered context
            node_context_extracted, combined_context_str = retrieve_bte_context(
                question=question,
                id=id,
                name=name,
                embedding_function=embedding_function,
                context_sim_threshold=0,
                context_sim_min_threshold=0.0,
                context_volume=None  
            )

            # Compress both
            node_ctx_compressed = zlib.compress(json.dumps(node_context_extracted).encode("utf-8"))
            combined_ctx_compressed = zlib.compress(json.dumps(combined_context_str).encode("utf-8"))

        except Exception as e:
            print(f"Error retrieving context for {id}: {e}")
            node_context_extracted = combined_context_str = None
            node_ctx_compressed = combined_ctx_compressed = None

        row_dict = {
            "Question": question,
            "extracted_disease": name,
            "curie_id": id,
            "Goldstandard": gold,
            "node_context_extracted": node_context_extracted,
            "combined_context_str": combined_context_str,
            "node_context_extracted_compressed": node_ctx_compressed,
            "combined_context_str_compressed": combined_ctx_compressed
        }

        out_df = pd.concat([out_df, pd.DataFrame([row_dict])], ignore_index=True)
        out_df.to_csv(save_path, index=False)
        processed.add(question)
        print(f"Saved context for: {name}")

    print(f"All done! Contexts saved to → {save_path}")
    return out_df


In [111]:
embedding_function

HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 350, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 768, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False})
), model_name='pritamdeka/S-PubMedBert-MS-MARCO', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False)

In [91]:
merged_df.head(1)

,Model,Module,Question,Goldstandard,extracted_disease,curie_id
0,GeneGPT,Gene disease association,The name of the gene related to Hemolytic anemia due to phosphofructokinase deficiency is,PFKL,Hemolytic anemia due to phosphofructokinase deficiency,MONDO:0009113


In [112]:
merged_df["Question_new"] = merged_df["extracted_disease"].apply(
    lambda disease: f"What is the name of the gene related to {disease}?"
)

In [95]:
merged_df.head(1)

,Model,Module,Question,Goldstandard,extracted_disease,curie_id,Question_new
0,GeneGPT,Gene disease association,The name of the gene related to Hemolytic anemia due to phosphofructokinase deficiency is,PFKL,Hemolytic anemia due to phosphofructokinase deficiency,MONDO:0009113,What is the name of the gene related to Hemolytic anemia due to phosphofructokinase deficiency?


In [113]:
save_path = "data/geneTuring/gene_dis_assoc_context_results_df.csv"
results_df = build_bte_context_df(
    df=merged_df,
    embedding_function=embedding_function,
    save_path=save_path
)

Processing id's:   0%|                                        | 0/100 [00:00<?, ?it/s]

Question: What is the name of the gene related to Pseudohypoparathyroidism Ic?
Processing ID: MONDO:0019992


Processing id's:   3%|▉                               | 3/100 [00:00<00:17,  5.51it/s]

Saved context for: Pseudohypoparathyroidism Ic
Question: What is the name of the gene related to Glycine N-methyltransferase deficiency?
Processing ID: MONDO:0011698
Saved context for: Glycine N-methyltransferase deficiency
Question: What is the name of the gene related to Meesmann corneal dystrophy?
Processing ID: MONDO:0007379


Processing id's:   6%|█▉                              | 6/100 [00:00<00:15,  6.14it/s]

...

Saved context for: Hartsfield syndrome
Question: What is the name of the gene related to Hodgkin lymphoma?
Processing ID: MONDO:0018908


Processing id's:  99%|██████████████████████████████▋| 99/100 [00:24<00:00,  1.32it/s]

Saved context for: Hodgkin lymphoma
Question: What is the name of the gene related to Hemorrhagic diathesis due to antithrombin Pittsburgh?
Processing ID: MONDO:0008802
No context found for id; skipping similarity.


Processing id's: 100%|██████████████████████████████| 100/100 [00:24<00:00,  4.08it/s]

Saved context for: Hemorrhagic diathesis due to antithrombin Pittsburgh
All done! Contexts saved to → data/geneTuring/gene_dis_assoc_context_results_df.csv


In [68]:
results_df = pd.read_csv("data/geneTuring/gene_dis_assoc_context_results_df.csv")
results_df.columns

Index(['Question', 'extracted_disease', 'curie_id', 'Goldstandard',
       'node_context_extracted', 'combined_context_str',
       'node_context_extracted_compressed', 'combined_context_str_compressed'],
      dtype='object')

In [69]:
results_df.shape

(100, 8)

In [70]:
results_df[['Question', 'extracted_disease', 'curie_id', 'Goldstandard', 'combined_context_str']].tail(2)

Question  \
98                                      What is the name of the gene related to Hodgkin lymphoma?   
99  What is the name of the gene related to Hemorrhagic diathesis due to antithrombin Pittsburgh?   

                                       extracted_disease       curie_id  \
98                                      Hodgkin lymphoma  MONDO:0018908   
99  Hemorrhagic diathesis due to antithrombin Pittsburgh  MONDO:0008802   

   Goldstandard  \
98      KLHDC8B   
99     SERPINA1   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             

In [71]:
results_df.columns

Index(['Question', 'extracted_disease', 'curie_id', 'Goldstandard',
       'node_context_extracted', 'combined_context_str',
       'node_context_extracted_compressed', 'combined_context_str_compressed'],
      dtype='object')

In [72]:
def count_empty_context(series):
    return (
        series.isna() |
        series.eq("") |
        series.eq("Context:\n\n")
    ).sum()

summary = {
    "node_context_extracted_empty": count_empty_context(results_df["node_context_extracted"]),
    "combined_context_str_empty": count_empty_context(results_df["combined_context_str"]),
    "total_rows": len(results_df)
}

summary

{'node_context_extracted_empty': 10,
 'combined_context_str_empty': 10,
 'total_rows': 100}

In [24]:
# empty_node_ctx = results_df[
#     results_df["node_context_extracted"].isna()
#     | results_df["node_context_extracted"].eq("")
#     | results_df["node_context_extracted"].eq("Context:\n\n")
# ]
# empty_node_ctx

# Prompt

## gpt-4o-mini

In [182]:
save_name = "gpt_4o_mini_PROMPT_geneTuring_gene_dis_assoc_context_output.csv"
SAVE_PATH = "data/geneTuring/results"
save_path = os.path.join(SAVE_PATH, save_name)
print("Saving to:", save_path)

Saving to: data/geneTuring/results/gpt_4o_mini_PROMPT_geneTuring_gene_dis_assoc_context_output.csv


In [183]:
SYSTEM_PROMPT = """You are an expert biomedical researcher. Please provide your answer (only gene name) in the following JSON format for the Question asked:
{
  "answer": <correct answer>
}


"""

In [184]:
CHAT_MODEL_ID = "gpt-4o-mini"
CHAT_DEPLOYMENT_ID = "gpt-4o-mini"
TEMPERATURE = config_data["LLM_TEMPERATURE"]
print("SYSTEM_PROMPT:", SYSTEM_PROMPT)
print("TEMPERATURE:", TEMPERATURE)
print("SAVE_PATH:", SAVE_PATH)

SYSTEM_PROMPT: You are an expert biomedical researcher. Please provide your answer (only gene name) in the following JSON format for the Question asked:
{
  "answer": <correct answer>
}



TEMPERATURE: 0
SAVE_PATH: data/geneTuring/results


In [180]:
results_df.columns

Index(['Question', 'extracted_disease', 'curie_id', 'Goldstandard',
       'node_context_extracted', 'combined_context_str',
       'node_context_extracted_compressed', 'combined_context_str_compressed'],
      dtype='object')

In [185]:
filtered_df = results_df[['Question', 'extracted_disease', 'curie_id', 'Goldstandard']]
filtered_df.shape

(100, 4)

In [186]:
start_time = time.time()
answer_list = []

for index, row in filtered_df.iterrows():
    question = "Question: " + str(row["Question"])
    print(f"Processing question {index + 1}/{len(filtered_df)}: {question}")
    try:
        output = get_GPT_response(question, SYSTEM_PROMPT, CHAT_MODEL_ID, CHAT_DEPLOYMENT_ID, temperature=TEMPERATURE)
        answer_list.append((row["Question"], row["extracted_disease"], row["curie_id"], row["Goldstandard"], output))
        print(f"Success: Processed question {index + 1}")
    except Exception as e:
        print(f"Error processing question {index + 1}: {e}")
        answer_list.append((row["Question"], row["extracted_disease"], row["curie_id"], row["Goldstandard"], "Error"))
        continue

answer_df = pd.DataFrame(answer_list, columns=["Question","extracted_disease","curie_id","Goldstandard","llm_answer_prompt_test"])
answer_df.to_csv(os.path.join(SAVE_PATH, save_name), index=False, header=True)
print("Completed in {:.2f} min".format((time.time() - start_time) / 60))


Processing question 1/100: Question: What is the name of the gene related to Hemolytic anemia due to phosphofructokinase deficiency?
Success: Processed question 1
Processing question 2/100: Question: What is the name of the gene related to Distal renal tubular acidosis?
Success: Processed question 2
Processing question 3/100: Question: What is the name of the gene related to Pseudohypoparathyroidism Ic?
Success: Processed question 3
Processing question 4/100: Question: What is the name of the gene related to Glycine N-methyltransferase deficiency?
Success: Processed question 4
Processing question 5/100: Question: What is the name of the gene related to Meesmann corneal dystrophy?
Success: Processed question 5
Processing question 6/100: Question: What is the name of the gene related to Chronic atrial and intestinal dysrhythmia?
Success: Processed question 6
Processing question 7/100: Question: What is the name of the gene related to Sensorineural deafness with mild renal dysfunction?
Su

In [73]:
gpt4omini_prompt = pd.read_csv("data/geneTuring/results/gpt_4o_mini_PROMPT_geneTuring_gene_dis_assoc_context_output.csv")
gpt4omini_prompt.shape

(100, 5)

In [74]:
gpt4omini_prompt.head()

,Question,extracted_disease,curie_id,Goldstandard,llm_answer_prompt_test
0,What is the name of the gene related to Hemolytic anemia due to phosphofructokinase deficiency?,Hemolytic anemia due to phosphofructokinase deficiency,MONDO:0009113,PFKL,"{\n ""answer"": ""PFKFB3""\n}"
1,What is the name of the gene related to Distal renal tubular acidosis?,Distal renal tubular acidosis,MONDO:0015827,"SLC4A1, ATP6V0A4","{\n ""answer"": ""SLC4A1""\n}"
2,What is the name of the gene related to Pseudohypoparathyroidism Ic?,Pseudohypoparathyroidism Ic,MONDO:0019992,GNAS,"{\n ""answer"": ""GNAS""\n}"
3,What is the name of the gene related to Glycine N-methyltransferase deficiency?,Glycine N-methyltransferase deficiency,MONDO:0011698,GNMT,"{\n ""answer"": ""GNMT""\n}"
4,What is the name of the gene related to Meesmann corneal dystrophy?,Meesmann corneal dystrophy,MONDO:0007379,"KRT12, KRT3","{\n ""answer"": ""KRT12""\n}"


In [75]:
def extract_gene_prediction(df, output_col="output_NC", new_col="predicted_gene"):

    def extract_answer(raw):
        if pd.isna(raw):
            return None
        # Clean markdown-style JSON blocks
        cleaned = str(raw).strip()
        cleaned = re.sub(r"^```(?:json)?", "", cleaned, flags=re.IGNORECASE).strip()
        cleaned = re.sub(r"```$", "", cleaned).strip()
        try:
            parsed = json.loads(cleaned)
            return parsed.get("answer", "").strip().upper()
        except Exception:
            return None

    df = df.copy()
    df[new_col] = df[output_col].apply(extract_answer)
    return df

In [76]:
gpt4omini_df = extract_gene_prediction(gpt4omini_prompt, output_col="llm_answer_prompt_test")


In [32]:
gpt4omini_df.head(1)

,Question,extracted_disease,curie_id,Goldstandard,llm_answer_prompt_test,predicted_gene
0,What is the name of the gene related to Hemolytic anemia due to phosphofructokinase deficiency?,Hemolytic anemia due to phosphofructokinase deficiency,MONDO:0009113,PFKL,"{\n ""answer"": ""PFKFB3""\n}",PFKFB3


In [77]:
def compare_predictions_to_gold(df, pred_col="predicted_gene", gold_col="Goldstandard"):
    df = df.copy()

    # Normalize both gold and predicted
    df["Goldstandard_norm"] = (
        df[gold_col]
        .astype(str)
        .str.upper()
        .str.replace(r"\s+", "", regex=True)
    )
    df["Pred_norm"] = (
        df[pred_col]
        .astype(str)
        .str.upper()
        .str.replace(r"\s+", "", regex=True)
    )

    # Correct if prediction appears within comma-separated gold list
    df["correct"] = df.apply(
        lambda row: (
            isinstance(row["Pred_norm"], str)
            and row["Pred_norm"] in row["Goldstandard_norm"].split(",")
        ),
        axis=1,
    )

    # Only include rows with predictions
    valid_rows = df[pred_col].notna()
    y_true = df.loc[valid_rows, "Goldstandard_norm"]
    y_pred = df.loc[valid_rows, pred_col]

    # Compute metrics (micro accuracy on correct flag)
    metrics = {
        "N": len(df),
        "Valid Predictions": valid_rows.sum(),
        "Correct": df["correct"].sum(),
        "Accuracy": df["correct"].mean(),  
        "Precision": precision_score(df["correct"], [True]*len(df), average='binary', zero_division=0)
        if df["correct"].any() else 0.0,
        "Recall": df["correct"].mean(),
        "F1 Score": f1_score(df["correct"], [True]*len(df), zero_division=0)
        if df["correct"].any() else 0.0,
    }

    return df, metrics

In [78]:
evaluated_df, metrics = compare_predictions_to_gold(gpt4omini_df,pred_col="predicted_gene")

print(" Evaluation Metrics:")
for k, v in metrics.items():
    print(f"{k}: {v:.3f}" if isinstance(v, float) else f"{k}: {v}")

 Evaluation Metrics:
N: 100
Valid Predictions: 100
Correct: 33
Accuracy: 0.330
Precision: 0.330
Recall: 0.330
F1 Score: 0.496


In [200]:
evaluated_df.head()

,Question,extracted_disease,curie_id,Goldstandard,llm_answer_prompt_test,predicted_gene,Goldstandard_norm,Pred_norm,correct
0,What is the name of the gene related to Hemolytic anemia due to phosphofructokinase deficiency?,Hemolytic anemia due to phosphofructokinase deficiency,MONDO:0009113,PFKL,"{\n ""answer"": ""PFKFB3""\n}",PFKFB3,PFKL,PFKFB3,False
1,What is the name of the gene related to Distal renal tubular acidosis?,Distal renal tubular acidosis,MONDO:0015827,"SLC4A1, ATP6V0A4","{\n ""answer"": ""SLC4A1""\n}",SLC4A1,"SLC4A1,ATP6V0A4",SLC4A1,True
2,What is the name of the gene related to Pseudohypoparathyroidism Ic?,Pseudohypoparathyroidism Ic,MONDO:0019992,GNAS,"{\n ""answer"": ""GNAS""\n}",GNAS,GNAS,GNAS,True
3,What is the name of the gene related to Glycine N-methyltransferase deficiency?,Glycine N-methyltransferase deficiency,MONDO:0011698,GNMT,"{\n ""answer"": ""GNMT""\n}",GNMT,GNMT,GNMT,True
4,What is the name of the gene related to Meesmann corneal dystrophy?,Meesmann corneal dystrophy,MONDO:0007379,"KRT12, KRT3","{\n ""answer"": ""KRT12""\n}",KRT12,"KRT12,KRT3",KRT12,True


## gpt-4o

In [201]:
save_name = "gpt_4o_PROMPT_geneTuring_gene_dis_assoc_context_output.csv"
SAVE_PATH = "data/geneTuring/results"
save_path = os.path.join(SAVE_PATH, save_name)
print("Saving to:", save_path)

Saving to: data/geneTuring/results/gpt_4o_PROMPT_geneTuring_gene_dis_assoc_context_output.csv


In [202]:
SYSTEM_PROMPT = """You are an expert biomedical researcher. Please provide your answer (only gene name) in the following JSON format for the Question asked:
{
  "answer": <correct answer>
}


"""

In [203]:
CHAT_MODEL_ID = "gpt-4o"
CHAT_DEPLOYMENT_ID = "gpt-4o"
TEMPERATURE = config_data["LLM_TEMPERATURE"]
print("SYSTEM_PROMPT:", SYSTEM_PROMPT)
print("TEMPERATURE:", TEMPERATURE)
print("SAVE_PATH:", SAVE_PATH)

SYSTEM_PROMPT: You are an expert biomedical researcher. Please provide your answer (only gene name) in the following JSON format for the Question asked:
{
  "answer": <correct answer>
}



TEMPERATURE: 0
SAVE_PATH: data/geneTuring/results


In [204]:
start_time = time.time()
answer_list = []

for index, row in filtered_df.iterrows():
    question = "Question: " + str(row["Question"])
    print(f"Processing question {index + 1}/{len(filtered_df)}: {question}")
    try:
        output = get_GPT_response(question, SYSTEM_PROMPT, CHAT_MODEL_ID, CHAT_DEPLOYMENT_ID, temperature=TEMPERATURE)
        answer_list.append((row["Question"], row["extracted_disease"], row["curie_id"], row["Goldstandard"], output))
        print(f"Success: Processed question {index + 1}")
    except Exception as e:
        print(f"Error processing question {index + 1}: {e}")
        answer_list.append((row["Question"], row["extracted_disease"], row["curie_id"], row["Goldstandard"], "Error"))
        continue

answer_df = pd.DataFrame(answer_list, columns=["Question","extracted_disease","curie_id","Goldstandard","llm_answer_prompt_test"])
answer_df.to_csv(os.path.join(SAVE_PATH, save_name), index=False, header=True)
print("Completed in {:.2f} min".format((time.time() - start_time) / 60))

Processing question 1/100: Question: What is the name of the gene related to Hemolytic anemia due to phosphofructokinase deficiency?
Success: Processed question 1
Processing question 2/100: Question: What is the name of the gene related to Distal renal tubular acidosis?
Success: Processed question 2
Processing question 3/100: Question: What is the name of the gene related to Pseudohypoparathyroidism Ic?
Success: Processed question 3
Processing question 4/100: Question: What is the name of the gene related to Glycine N-methyltransferase deficiency?
Success: Processed question 4
Processing question 5/100: Question: What is the name of the gene related to Meesmann corneal dystrophy?
Success: Processed question 5
Processing question 6/100: Question: What is the name of the gene related to Chronic atrial and intestinal dysrhythmia?
Success: Processed question 6
Processing question 7/100: Question: What is the name of the gene related to Sensorineural deafness with mild renal dysfunction?
Su

In [79]:
gpt4o_prompt = pd.read_csv("data/geneTuring/results/gpt_4o_PROMPT_geneTuring_gene_dis_assoc_context_output.csv")
gpt4o_prompt.shape

(100, 5)

In [80]:
gpt4o_df = extract_gene_prediction(gpt4o_prompt, output_col="llm_answer_prompt_test")


In [81]:
evaluated_df, metrics = compare_predictions_to_gold(gpt4o_df,pred_col="predicted_gene")

print(" Evaluation Metrics:")
for k, v in metrics.items():
    print(f"{k}: {v:.3f}" if isinstance(v, float) else f"{k}: {v}")

 Evaluation Metrics:
N: 100
Valid Predictions: 100
Correct: 56
Accuracy: 0.560
Precision: 0.560
Recall: 0.560
F1 Score: 0.718


# BTE

## gpt-40mini

In [141]:
save_name = "gpt_4o_mini_BTE-RAG_geneTuring_gene_dis_assoc_context_output.csv"
SAVE_PATH = "data/geneTuring/results"
save_path = os.path.join(SAVE_PATH, save_name)
print("Saving to:", save_path)
#context_str = row["node_context_extracted"]

Saving to: data/geneTuring/results/gpt_4o_mini_BTE-RAG_geneTuring_gene_disease_assoc_context_output.csv


In [142]:
df = results_df.copy()

# Add columns to hold LLM outputs
df["output_NC"] = None                
df["combined_trunc"] = None          


llm_type = "gpt-4o-mini"

system_prompt = """You are an advanced biomedical AI assistant. Use your most recent knowledge in addition to the Context provided when needed to answer accurately. 
Answer with the gene symbol only.

*Answer Format*: Provide your answer (just the gene symbol) in the following JSON format:
{
  "answer": "<GENE_SYMBOL>"
}
"""


start = time.time()

for idx, row in df.iterrows():
    question = row["Question"]
    context_str = row["combined_context_str"]
    # context_str = row["node_context_extracted"]
    
    
    try:
        output, trunc = retrieve_combined_from_llm(
            question=question,
            combined_context_str=context_str,
            llm_type=llm_type,
            system_prompt=system_prompt
        )
        df.at[idx, "output_NC"] = output
        df.at[idx, "combined_trunc"] = trunc

    except Exception as e:
        print(f"[Row {idx}] LLM ERROR:", e)
        df.at[idx, "output_NC"] = None
        df.at[idx, "combined_trunc"] = None

    # Progress update
    if (idx + 1) % 10 == 0:
        elapsed = (time.time() - start) / 60
        print(f"Processed {idx+1}/{len(df)} rows — {elapsed:.1f} min elapsed")


os.makedirs(SAVE_PATH, exist_ok=True)
df.to_csv(save_path, index=False)
print(f"All done — results saved to {save_path}")


Original token count: 151
Original token count: 935
Original token count: 3723
Original token count: 110
Original token count: 546
Original token count: 399
Original token count: 105
Original token count: 2
Original token count: 92
Original token count: 287
Processed 10/100 rows — 0.1 min elapsed
Original token count: 2
Original token count: 259
Original token count: 164
Original token count: 60
Original token count: 419
Original token count: 112
Original token count: 1813
Original token count: 1083
Original token count: 39
Original token count: 75
Processed 20/100 rows — 0.2 min elapsed
Original token count: 2
Original token count: 52
Original token count: 347
Original token count: 319
Original token count: 1498
Original token count: 30
Original token count: 566
Original token count: 7158
Original token count: 67
Original token count: 3554
Processed 30/100 rows — 0.3 min elapsed
Original token count: 2
Original token count: 315
Original token count: 2
Original token count: 251
Origina

In [82]:
gpt4omini_df = pd.read_csv("data/geneTuring/results/gpt_4o_mini_BTE-RAG_geneTuring_gene_dis_assoc_context_output.csv")
gpt4omini_df.columns

Index(['Question', 'extracted_disease', 'curie_id', 'Goldstandard',
       'node_context_extracted', 'combined_context_str',
       'node_context_extracted_compressed', 'combined_context_str_compressed',
       'output_NC', 'combined_trunc'],
      dtype='object')

In [83]:
gpt4omini_df = extract_gene_prediction(gpt4omini_df, output_col="output_NC")

In [84]:
evaluated_df, metrics = compare_predictions_to_gold(gpt4omini_df)


In [85]:
evaluated_df, metrics = compare_predictions_to_gold(gpt4omini_df)

print(" Evaluation Metrics:")
for k, v in metrics.items():
    print(f"{k}: {v:.3f}" if isinstance(v, float) else f"{k}: {v}")


 Evaluation Metrics:
N: 100
Valid Predictions: 100
Correct: 77
Accuracy: 0.770
Precision: 0.770
Recall: 0.770
F1 Score: 0.870


In [54]:
# empty_context_correct = evaluated_df[
#     (evaluated_df["combined_context_str"].str.strip() == "Context:") &
#     (evaluated_df["correct"] == True)
# ]


# empty_context_correct[[
#     "Question", "extracted_disease", "curie_id", "Goldstandard", "predicted_gene", "correct", "combined_context_str"
# ]].shape

In [55]:
# empty_context_correct[[
#     "Question", "extracted_disease", "curie_id", "Goldstandard", "predicted_gene", "correct", "combined_context_str"
# ]]

## gpt-4o

In [166]:
save_name = "gpt_4o_BTE-RAG_geneTuring_gene_dis_assoc_context_output.csv"
SAVE_PATH = "data/geneTuring/results"
save_path = os.path.join(SAVE_PATH, save_name)
print("Saving to:", save_path)

Saving to: data/geneTuring/results/gpt_4o_BTE-RAG_geneTuring_gene_dis_assoc_context_output.csv


In [167]:
df = results_df.copy()

# Add columns to hold LLM outputs
df["output_NC"] = None                
df["combined_trunc"] = None          


llm_type = "gpt-4o"

system_prompt = """You are an advanced biomedical AI assistant. Use your most recent knowledge in addition to the Context provided when needed to answer accurately. 
Answer with the gene symbol only.

*Answer Format*: Provide your answer (just the gene symbol) in the following JSON format:
{
  "answer": "<GENE_SYMBOL>"
}
"""


start = time.time()

for idx, row in df.iterrows():
    question = row["Question"]
    context_str = row["combined_context_str"]

    try:
        output, trunc = retrieve_combined_from_llm(
            question=question,
            combined_context_str=context_str,
            llm_type=llm_type,
            system_prompt=system_prompt
        )
        df.at[idx, "output_NC"] = output
        df.at[idx, "combined_trunc"] = trunc

    except Exception as e:
        print(f"[Row {idx}] LLM ERROR:", e)
        df.at[idx, "output_NC"] = None
        df.at[idx, "combined_trunc"] = None

    # Progress update
    if (idx + 1) % 10 == 0:
        elapsed = (time.time() - start) / 60
        print(f"Processed {idx+1}/{len(df)} rows — {elapsed:.1f} min elapsed")


os.makedirs(SAVE_PATH, exist_ok=True)
df.to_csv(save_path, index=False)
print(f"All done — results saved to {save_path}")

Original token count: 151
Original token count: 935
Original token count: 3723
Original token count: 110
Original token count: 546
Original token count: 399
Original token count: 105
Original token count: 2
Original token count: 92
Original token count: 287
Processed 10/100 rows — 0.1 min elapsed
Original token count: 2
Original token count: 259
Original token count: 164
Original token count: 60
Original token count: 419
Original token count: 112
Original token count: 1813
Original token count: 1083
Original token count: 39
Original token count: 75
Processed 20/100 rows — 0.2 min elapsed
Original token count: 2
Original token count: 52
Original token count: 347
Original token count: 319
Original token count: 1498
Original token count: 30
Original token count: 566
Original token count: 7158
Original token count: 67
Original token count: 3554
Processed 30/100 rows — 0.3 min elapsed
Original token count: 2
Original token count: 315
Original token count: 2
Original token count: 251
Origina

In [56]:
gpt4o_df = pd.read_csv("data/geneTuring/results/gpt_4o_BTE-RAG_geneTuring_gene_dis_assoc_context_output.csv")
gpt4o_df.columns

Index(['Question', 'extracted_disease', 'curie_id', 'Goldstandard',
       'node_context_extracted', 'combined_context_str',
       'node_context_extracted_compressed', 'combined_context_str_compressed',
       'output_NC', 'combined_trunc'],
      dtype='object')

In [57]:
gpt4o_df = extract_gene_prediction(gpt4o_df, output_col="output_NC")


In [58]:
evaluated_df, metrics = compare_predictions_to_gold(gpt4o_df)

print(" Evaluation Metrics:")
for k, v in metrics.items():
    print(f"{k}: {v:.3f}" if isinstance(v, float) else f"{k}: {v}")

 Evaluation Metrics:
N: 100
Valid Predictions: 100
Correct: 80
Accuracy: 0.800
Precision: 0.800
Recall: 0.800
F1 Score: 0.889
